# Setup

In [ ]:
# set gemini key in the environment
from google.colab import userdata
import os

api_key = userdata.get('Gemini_API_KEY_RAG')
os.environ['GEMINI_API_KEY'] = api_key


In [ ]:
from google import genai
from IPython.display import Markdown

client = genai.Client()

In [ ]:
%cd /content/drive/MyDrive/RAG/RAG/RAG with Gemini File Search

/content/drive/MyDrive/RAG/RAG/RAG with Gemini File Search


# Create the Knowledge Component

In [9]:
# Define place where the dos are and extensions
docs_directory = "Docs"
supported_extensions = [".txt", ".py"]

In [17]:
from genericpath import isfile
# append all file names and path together
files_path = []

for filename in os.listdir(docs_directory):
  file_path = os.path.join(docs_directory, filename)
  if os.path.isfile(file_path):
    file_extension = os.path.splitext(filename)[1].lower()
    if file_extension in supported_extensions:
      files_path.append(file_path)

print(f"Files path {files_path}")


Files path ['Docs/7.09 Introduction to Contrastive Learning.txt', 'Docs/7.07 Python - Transcribing Audio with OpenAI Whisper.txt', 'Docs/7.10 Understanding the CLIP Model.txt', 'Docs/7.03 Python - Sep up and Video.txt', 'Docs/7.20 Python - GenAI Model Setup.txt', 'Docs/7.16 Python - Visualizing Text and Images.txt', 'Docs/7.01 Game Plan for Multimodal RAG.txt', 'Docs/7.02 Introduction to Multimodal RAG.txt', 'Docs/7.17 Python - Query Embedding.txt', 'Docs/7.18 Python - Cosine Similarity of Query and Text.txt', 'Docs/7.22 What did we learn in this section.txt', 'Docs/7.07 Whisper Model.txt', 'Docs/7.08 Python - Extracting Frames.txt', 'Docs/7.12 Python - Chunking and Embedding Text.txt', 'Docs/7.15 Python - Contrastive Learning and Cosine Similarity.txt', 'Docs/7.14 Understanding Cosine Similarity in Multimodal Contexts.txt', 'Docs/7.11 Python - Tokenize text.txt', 'Docs/7.21 Python - GenAI model.txt', 'Docs/7.13 Python - Embedding Images.txt', 'Docs/7.05 Python - Compressing Audio.txt'

In [18]:
# Creating a file search store
file_search_store = client.file_search_stores.create(
    config={
        "display_name": "mini rubber ducky",
        "embedding_model": "models/gemini-embedding-2"
    }
)

In [23]:
# Retrieve the name & id
for store in client.file_search_stores.list():
  store_id = store.name.split('/')[-1]   # extract ID from full resource name
  print(f"Name: {store.display_name} | ID: {store_id}")

my_file_search_store = client.file_search_stores.get(name=file_search_store.name)

Name: mini rubber ducky | ID: mini-rubber-ducky-gpbmz1evc4i5


In [36]:
import time

# Upoad files to gemini file search store
document_names = []

for file_path in files_path:
  operation = client.file_search_stores.upload_to_file_search_store(
      file=file_path,
      file_search_store_name=my_file_search_store.name
  )

  # Gemini uploads are ASYNC — must wait for indexing to finish
  while not operation.done:
    time.sleep(5)
    operation = client.operations.get(operation)

  print(f"✅ {file_path} indexed")
  document_names.append(operation.response)


✅ Docs/7.09 Introduction to Contrastive Learning.txt indexed
✅ Docs/7.07 Python - Transcribing Audio with OpenAI Whisper.txt indexed
✅ Docs/7.10 Understanding the CLIP Model.txt indexed
✅ Docs/7.03 Python - Sep up and Video.txt indexed
✅ Docs/7.20 Python - GenAI Model Setup.txt indexed
✅ Docs/7.16 Python - Visualizing Text and Images.txt indexed
✅ Docs/7.01 Game Plan for Multimodal RAG.txt indexed
✅ Docs/7.02 Introduction to Multimodal RAG.txt indexed
✅ Docs/7.17 Python - Query Embedding.txt indexed
✅ Docs/7.18 Python - Cosine Similarity of Query and Text.txt indexed
✅ Docs/7.22 What did we learn in this section.txt indexed
✅ Docs/7.07 Whisper Model.txt indexed
✅ Docs/7.08 Python - Extracting Frames.txt indexed
✅ Docs/7.12 Python - Chunking and Embedding Text.txt indexed
✅ Docs/7.15 Python - Contrastive Learning and Cosine Similarity.txt indexed
✅ Docs/7.14 Understanding Cosine Similarity in Multimodal Contexts.txt indexed
✅ Docs/7.11 Python - Tokenize text.txt indexed
✅ Docs/7.21 Pyth

In [38]:
for doc in client.file_search_stores.documents.list(parent=my_file_search_store.name):
  print(doc.display_name, "|", doc.name)

7.09 Introduction to Contrastive Learning.txt | fileSearchStores/mini-rubber-ducky-gpbmz1evc4i5/documents/709-introduction-to-contras-wyjwmshuio3n
7.07 Python - Transcribing Audio with OpenAI Whisper.txt | fileSearchStores/mini-rubber-ducky-gpbmz1evc4i5/documents/707-python-transcribing-aud-z7gscuajt9ql
7.10 Understanding the CLIP Model.txt | fileSearchStores/mini-rubber-ducky-gpbmz1evc4i5/documents/710-understanding-the-clip--nk88pzwe9xw9
7.03 Python - Sep up and Video.txt | fileSearchStores/mini-rubber-ducky-gpbmz1evc4i5/documents/703-python-sep-up-and-video-8s3c1pmhzpr8
7.20 Python - GenAI Model Setup.txt | fileSearchStores/mini-rubber-ducky-gpbmz1evc4i5/documents/720-python-genai-model-setu-l2f1j8fn16sm
7.16 Python - Visualizing Text and Images.txt | fileSearchStores/mini-rubber-ducky-gpbmz1evc4i5/documents/716-python-visualizing-text-yn7pxl2qdumx
7.01 Game Plan for Multimodal RAG.txt | fileSearchStores/mini-rubber-ducky-gpbmz1evc4i5/documents/701-game-plan-for-multimoda-mhjciwbbya

# Rag System

In [49]:
# Call the API
interaction = client.interactions.create(
    model="gemini-3.5-flash-lite",
    system_instruction=(
        "You are a RAG assistant. Answer ONLY using information retrieved "
        "from the file search store. If the answer is not present in the "
        "retrieved documents, reply exactly: 'I could not find that in the "
        "provided documents.' Do not use outside knowledge. Do not guess."
    ),
    input = "When should build a multimodal RAG?",
    tools=[{
        "type": "file_search",
        "file_search_store_names": [my_file_search_store.name]
    }]
)

In [50]:
# Explore the output
display(Markdown(interaction.output_text))

You should build a multimodal RAG when your data and use case require retrieving and processing information across multiple data types or formats (such as text documents, images, audio clips, and videos) rather than just text alone. 

Specific scenarios include:
* **Working with diverse data sources:** When you need to synthesize insights from a mix of sources like text product specs, competitor images, audio clips from focus groups, and customer feedback videos.
* **Avoiding manual, time-consuming tasks:** When manually sifting through and extracting key points across different media formats is inefficient and you need a tool to pull insights quickly and cohesively.
* **Comprehensive understanding:** When a complete, unified narrative or report requires combining different modalities—such as coupling design elements from photos with insights from videos and text—to offer a full understanding of a topic (like a market strategy or product launch).

In [51]:
# Validaing my file usage
from google.genai import types

for step in interaction.steps:
    if step.type == "model_output":
        for content_block in step.content:
            if content_block.type == "text":
                print(content_block.text)
                if content_block.annotations:
                    print("\nSources:")
                    for annotation in content_block.annotations:
                        if annotation.type == "file_citation":
                            print(f"  - {annotation.file_name}: {annotation.source}")

You should build a multimodal RAG when your data and use case require retrieving and processing information across multiple data types or formats (such as text documents, images, audio clips, and videos) rather than just text alone. 

Specific scenarios include:
* **Working with diverse data sources:** When you need to synthesize insights from a mix of sources like text product specs, competitor images, audio clips from focus groups, and customer feedback videos.
* **Avoiding manual, time-consuming tasks:** When manually sifting through and extracting key points across different media formats is inefficient and you need a tool to pull insights quickly and cohesively.
* **Comprehensive understanding:** When a complete, unified narrative or report requires combining different modalities—such as coupling design elements from photos with insights from videos and text—to offer a full understanding of a topic (like a market strategy or product launch).

Sources:
  - 7.02 Introduction to Mult